# 🐦 BirdCLEF+ 2026 — Custom EfficientNet-B0 Training (Colab)

**Author:** Anuj Dev Singh · [codewithanuj.com](https://codewithanuj.com) · [@anujdevsingh](https://github.com/anujdevsingh)

**Purpose:** Train a custom EfficientNet-B0 on BirdCLEF 2026 training audio (Xeno-Canto recordings + Pantanal soundscapes), export to ONNX, and integrate as a new ensemble branch into the EoS6-bz inference notebook on Kaggle.

**Realistic target:** +0.003 to +0.008 LB over the current 0.949 → final score 0.952-0.957.

**Why this approach:** The existing EoS6-bz ensemble uses Perch v2 (Transformer), ProtoSSM (SSM), SED (CNN). Adding an EfficientNet-B0 trained on raw Xeno-Canto recordings adds genuine data-level diversity that the current pipeline doesn't see.

---

## ⚠️ Important — read before starting

1. **You must accept BirdCLEF 2026 competition rules first**: https://www.kaggle.com/competitions/birdclef-2026/rules
2. Free Colab T4 sessions disconnect after ~12 hours. **All checkpoints save to Google Drive** so you can resume.
3. Plan: train Fold 0 first (~3 hours). Validate it produces a good model. Then run Folds 1-4 across multiple Colab sessions.
4. If anything breaks: check the troubleshooting cells at the bottom.

---

## 📋 Pipeline overview

```
Kaggle API → Download data → Cache mel-spectrograms (HDF5) →
  Train EfficientNet-B0 (5 folds, with checkpointing) →
  Export ONNX → Upload to Kaggle as dataset →
  Integrate into EoS6-bz notebook → Submit
```


## 🔧 Step 1 — GPU check & install packages

In [ ]:
# Confirm we have a GPU. Should show Tesla T4, A100, or V100.
# If "No GPU available", go to Runtime → Change runtime type → Hardware accelerator → GPU.
!nvidia-smi

Sun May 24 10:32:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Install required packages (~2 min)
!pip install -q timm==1.0.11 torchaudio soundfile librosa pandas tqdm kaggle h5py scikit-learn onnx onnxruntime
print("✅ Packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.6 MB/s eta 0:00:00
✅ Packages installed


## 💾 Step 2 — Mount Google Drive for persistence

Critical for free Colab. All checkpoints + processed data save to your Drive so training survives session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Working directory in Drive
import os
WORK_DIR = '/content/drive/MyDrive/birdclef2026_anuj'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORK_DIR}/onnx', exist_ok=True)
os.makedirs(f'{WORK_DIR}/logs', exist_ok=True)
print(f'✅ Drive mounted. Working dir: {WORK_DIR}')
!df -h /content/drive/MyDrive | head -2

Mounted at /content/drive
✅ Drive mounted. Working dir: /content/drive/MyDrive/birdclef2026_anuj
Filesystem      Size  Used Avail Use% Mounted on
drive           113G   51G   62G  46% /content/drive


## 🔑 Step 3 — Kaggle API setup

To download competition data and later upload your trained weights as a dataset.

**Get your kaggle.json:**
1. Go to https://www.kaggle.com/settings/account
2. Scroll to "API" section
3. Click "Create New API Token" → downloads `kaggle.json`
4. Run cell below and upload that file

In [ ]:
from google.colab import files
import os

# Check if already uploaded to Drive (for resume)
kaggle_drive_path = f'{WORK_DIR}/kaggle.json'
if os.path.exists(kaggle_drive_path):
    print(f'Using existing kaggle.json from Drive')
    !cp $kaggle_drive_path ~/.kaggle/kaggle.json 2>/dev/null || (mkdir -p ~/.kaggle && cp $kaggle_drive_path ~/.kaggle/kaggle.json)
else:
    print('Upload your kaggle.json now...')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !cp ~/.kaggle/kaggle.json $kaggle_drive_path  # backup for resume

!chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions list 2>&1 | head -3
print('\n✅ Kaggle API configured')

Upload your kaggle.json now...


Saving kaggle.json to kaggle.json
ref                                                                              deadline             category         reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge      2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  

✅ Kaggle API configured


## 📥 Step 4 — Download BirdCLEF 2026 data

**Size**: ~6-10 GB. Takes 5-15 min on Colab.

The download goes to `/content/data` (Colab local, fast SSD). After preprocessing we cache spectrograms to Drive for persistence.

In [ ]:
import os
DATA_DIR = '/content/data'

# Skip if already downloaded (resume-friendly)
if os.path.exists(f'{DATA_DIR}/train.csv') or os.path.exists(f'{DATA_DIR}/train_audio'):
    print(f'✅ Data already exists at {DATA_DIR}')
else:
    !mkdir -p $DATA_DIR
    print('Downloading BirdCLEF 2026 dataset (~6-10 GB, 5-15 min)...')
    !kaggle competitions download -c birdclef-2026 -p $DATA_DIR
    print('Unzipping...')
    !unzip -q $DATA_DIR/birdclef-2026.zip -d $DATA_DIR/
    !rm $DATA_DIR/birdclef-2026.zip
    print('✅ Data ready')

!ls $DATA_DIR | head -20
!echo '---'
!du -sh $DATA_DIR

100% 15.0G/15.0G [14:46<00:00, 18.1MB/s]

Unzipping...
✅ Data ready
recording_location.txt
sample_submission.csv
taxonomy.csv
test_soundscapes
train_audio
train.csv
train_soundscapes
train_soundscapes_labels.csv
---
16G	/content/data


In [ ]:
# Inspect data structure to confirm what we have
import pandas as pd
from pathlib import Path

data_root = Path(DATA_DIR)
print('Top-level files:')
for f in sorted(data_root.iterdir()):
    if f.is_file():
        sz = f.stat().st_size / 1024 / 1024
        print(f'  {f.name:40s}  {sz:8.1f} MB')
    else:
        n_items = len(list(f.iterdir()))
        print(f'  {f.name:40s}  {n_items} items')

print('\nLooking for training metadata:')
for candidate in ['train.csv', 'train_metadata.csv']:
    p = data_root / candidate
    if p.exists():
        df = pd.read_csv(p)
        print(f'  ✅ Found {candidate}: shape={df.shape}')
        print(f'     Columns: {list(df.columns)}')
        print(df.head(3))
        break

Top-level files:
  recording_location.txt                         0.0 MB
  sample_submission.csv                          0.0 MB
  taxonomy.csv                                   0.0 MB
  test_soundscapes                          1 items
  train.csv                                      6.5 MB
  train_audio                               206 items
  train_soundscapes                         10658 items
  train_soundscapes_labels.csv                   0.1 MB

Looking for training metadata:
  ✅ Found train.csv: shape=(35549, 15)
     Columns: ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']
  primary_label secondary_labels type  latitude  longitude scientific_name  \
0       1161364               []   []  -22.7562   -46.8666    Guyalna cuta   
1       1161364               []   []  -22.7558   -46.8700    Guyalna cuta   
2       1161364       

## ⚙️ Step 5 — Configuration

All hyperparameters in one place. Defaults are proven from BirdCLEF 2024-2025 winning recipes.

In [ ]:
# Master configuration
CFG = {
    # Audio
    'SR': 32000,
    'WINDOW_SEC': 5,
    'WINDOW_SAMPLES': 32000 * 5,

    # Mel-spectrogram
    'N_MELS': 128,
    'N_FFT': 2048,
    'HOP_LENGTH': 512,
    'FMIN': 50,
    'FMAX': 14000,
    'TOP_DB': 80,
    'IMG_SIZE': 224,  # final resize for image classifier

    # Model
    'MODEL_NAME': 'tf_efficientnet_b0.ns_jft_in1k',  # timm name

    # Training
    'BATCH_SIZE': 64,
    'EPOCHS': 30,
    'LR': 1e-3,
    'WEIGHT_DECAY': 1e-3,
    'WARMUP_EPOCHS': 3,
    'PATIENCE': 8,
    'GRAD_CLIP': 1.0,

    # Augmentation
    'MIXUP_PROB': 0.5,
    'MIXUP_ALPHA': 0.4,
    'SPECAUG_FREQ_MASK': 16,
    'SPECAUG_TIME_MASK': 32,

    # Cross-validation
    'N_FOLDS': 5,
    'SEED': 42,

    # Paths
    'DATA_DIR': '/content/data',
    'WORK_DIR': WORK_DIR,
    'CACHE_H5': f'{WORK_DIR}/melspec_cache.h5',
    'CHECKPOINTS': f'{WORK_DIR}/checkpoints',
    'ONNX_DIR': f'{WORK_DIR}/onnx',
    'LOG_DIR': f'{WORK_DIR}/logs',

    # Class limit per species (BirdCLEF tradition — cap imbalanced data)
    'MAX_CLIPS_PER_SPECIES': 500,
}

# Pretty print
import json as _json
print(_json.dumps({k: v for k, v in CFG.items() if not isinstance(v, str) or len(str(v)) < 80}, indent=2, default=str))

{
  "SR": 32000,
  "WINDOW_SEC": 5,
  "WINDOW_SAMPLES": 160000,
  "N_MELS": 128,
  "N_FFT": 2048,
  "HOP_LENGTH": 512,
  "FMIN": 50,
  "FMAX": 14000,
  "TOP_DB": 80,
  "IMG_SIZE": 224,
  "MODEL_NAME": "tf_efficientnet_b0.ns_jft_in1k",
  "BATCH_SIZE": 64,
  "EPOCHS": 30,
  "LR": 0.001,
  "WEIGHT_DECAY": 0.001,
  "WARMUP_EPOCHS": 3,
  "PATIENCE": 8,
  "GRAD_CLIP": 1.0,
  "MIXUP_PROB": 0.5,
  "MIXUP_ALPHA": 0.4,
  "SPECAUG_FREQ_MASK": 16,
  "SPECAUG_TIME_MASK": 32,
  "N_FOLDS": 5,
  "SEED": 42,
  "DATA_DIR": "/content/data",
  "WORK_DIR": "/content/drive/MyDrive/birdclef2026_anuj",
  "CACHE_H5": "/content/drive/MyDrive/birdclef2026_anuj/melspec_cache.h5",
  "CHECKPOINTS": "/content/drive/MyDrive/birdclef2026_anuj/checkpoints",
  "ONNX_DIR": "/content/drive/MyDrive/birdclef2026_anuj/onnx",
  "LOG_DIR": "/content/drive/MyDrive/birdclef2026_anuj/logs",
  "MAX_CLIPS_PER_SPECIES": 500
}


## 📚 Step 6 — Imports & seed everything

In [ ]:
import os, sys, gc, time, math, random, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import h5py
import soundfile as sf
import librosa
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import timm

# Deterministic seeds
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_all(CFG['SEED'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {device}')
print(f'timm {timm.__version__}')

Torch 2.10.0+cu128 | CUDA: True | Device: cuda
timm 1.0.11


## 📑 Step 7 — Load training metadata

BirdCLEF 2026 has `train_audio/{species_code}/{XC_id}.ogg` files plus a `train.csv` or `train_metadata.csv`.

In [ ]:
# Auto-detect training metadata file
data_root = Path(CFG['DATA_DIR'])
train_csv = None
for candidate in ['train.csv', 'train_metadata.csv']:
    p = data_root / candidate
    if p.exists():
        train_csv = p
        break

assert train_csv is not None, f'Could not find train.csv in {data_root}'
print(f'Using metadata: {train_csv}')

train_df = pd.read_csv(train_csv)
print(f'Total clips: {len(train_df)}')
print(f'Columns: {list(train_df.columns)}')
train_df.head()

Using metadata: /content/data/train.csv
Total clips: 35549
Columns: ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat


In [ ]:
# Determine path column and species column
PATH_COL = None
LABEL_COL = None
for c in ['filename', 'filepath', 'file_path', 'path']:
    if c in train_df.columns: PATH_COL = c; break
for c in ['primary_label', 'species', 'label']:
    if c in train_df.columns: LABEL_COL = c; break

assert PATH_COL and LABEL_COL, f'Could not auto-detect columns. Available: {list(train_df.columns)}'
print(f'Path column: {PATH_COL}')
print(f'Label column: {LABEL_COL}')

# Load species taxonomy (defines our 234 target classes)
taxonomy = pd.read_csv(data_root / 'taxonomy.csv')
SPECIES = sorted(taxonomy['primary_label'].tolist())
N_CLASSES = len(SPECIES)
species_to_idx = {s: i for i, s in enumerate(SPECIES)}
print(f'\nTarget species count: {N_CLASSES}')

# Filter train_df to species in the taxonomy (drop cross-species junk if any)
train_df = train_df[train_df[LABEL_COL].isin(species_to_idx)].copy()
train_df['class_idx'] = train_df[LABEL_COL].map(species_to_idx)
print(f'Clips after taxonomy filter: {len(train_df)}')

# Distribution
class_counts = train_df[LABEL_COL].value_counts()
print(f'\nClass distribution: min={class_counts.min()}, max={class_counts.max()}, median={int(class_counts.median())}')
print(f'Species with <5 clips: {(class_counts < 5).sum()}')

Path column: filename
Label column: primary_label

Target species count: 234
Clips after taxonomy filter: 35549

Class distribution: min=1, max=499, median=125
Species with <5 clips: 14


In [ ]:
# Cap clips per species (BirdCLEF tradition: prevents over-represented species dominating)
MAX_PER = CFG['MAX_CLIPS_PER_SPECIES']
if MAX_PER and MAX_PER > 0:
    before = len(train_df)
    train_df = train_df.groupby(LABEL_COL, group_keys=False).apply(
        lambda g: g.sample(min(len(g), MAX_PER), random_state=CFG['SEED'])
    ).reset_index(drop=True)
    print(f'Capped at {MAX_PER}/species: {before} → {len(train_df)} clips')

# Resolve file paths
audio_dir = data_root / 'train_audio'
if (audio_dir / train_df.iloc[0][PATH_COL]).exists():
    # path column contains relative path including species
    train_df['full_path'] = train_df[PATH_COL].apply(lambda p: str(audio_dir / p))
else:
    # path column is just the filename — combine with species folder
    train_df['full_path'] = train_df.apply(lambda r: str(audio_dir / r[LABEL_COL] / r[PATH_COL]), axis=1)

# Verify a few exist
sample = train_df['full_path'].sample(5, random_state=42).tolist()
print('\nSample paths:')
for s in sample:
    print(f'  {s}  exists={os.path.exists(s)}')

Capped at 500/species: 35549 → 35549 clips

Sample paths:
  /content/data/train_audio/purjay1/XC844008.ogg  exists=True
  /content/data/train_audio/saytan1/iNat559494.ogg  exists=True
  /content/data/train_audio/thlwre1/XC562301.ogg  exists=True
  /content/data/train_audio/whbwar2/XC288893.ogg  exists=True
  /content/data/train_audio/whbwar2/iNat246929.ogg  exists=True


## 🎵 Step 8 — Precompute mel-spectrograms → HDF5 cache

We convert all training audio to 224×224 log-mel spectrograms ONCE and cache to disk. This is the most time-expensive step (~30-60 min) but only runs once.

**Strategy**: For each clip, take a single random 5s window (we'll vary the crop at training time via offsets). Save log-mel as int8 to keep cache small.

In [ ]:
def audio_to_logmel(y, cfg=CFG):
    """Compute log-mel spectrogram of a 5s waveform. Returns float32 [N_MELS, T]."""
    target = cfg['WINDOW_SAMPLES']
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    elif len(y) > target:
        # take random crop at preprocessing time → varies seed across runs
        start = np.random.randint(0, len(y) - target + 1)
        y = y[start:start + target]

    S = librosa.feature.melspectrogram(
        y=y, sr=cfg['SR'], n_fft=cfg['N_FFT'], hop_length=cfg['HOP_LENGTH'],
        n_mels=cfg['N_MELS'], fmin=cfg['FMIN'], fmax=cfg['FMAX'], power=2.0
    )
    S = librosa.power_to_db(S, top_db=cfg['TOP_DB']).astype(np.float32)
    # Normalize to [0, 1] approximately
    S = (S + cfg['TOP_DB']) / cfg['TOP_DB']
    return S  # shape (N_MELS, T)

def read_audio(path, sr=CFG['SR']):
    """Read full audio file, mix to mono."""
    y, file_sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if file_sr != sr:
        y = librosa.resample(y, orig_sr=file_sr, target_sr=sr)
    return y

In [ ]:
# Build the HDF5 cache (skip if already exists)
import h5py
CACHE_PATH = CFG['CACHE_H5']

if os.path.exists(CACHE_PATH):
    with h5py.File(CACHE_PATH, 'r') as f:
        print(f'✅ Cache already exists at {CACHE_PATH}')
        print(f'   Entries: {len(f.keys())}')
        sample_key = list(f.keys())[0]
        print(f'   Sample shape: {f[sample_key][:].shape}, dtype: {f[sample_key][:].dtype}')
else:
    print(f'Building cache → {CACHE_PATH}')
    print(f'Processing {len(train_df)} clips. This takes 30-60 min on free Colab...')

    failed = []
    seed_all(CFG['SEED'])

    with h5py.File(CACHE_PATH, 'w') as f:
        # Save metadata
        f.attrs['n_clips'] = len(train_df)
        f.attrs['n_classes'] = N_CLASSES

        for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Mel cache'):
            key = str(idx)
            if key in f:
                continue
            try:
                y = read_audio(row['full_path'])
                S = audio_to_logmel(y)
                # Store as int8 to save space (~75% reduction vs float32)
                S_int8 = np.clip(S * 127, -128, 127).astype(np.int8)
                ds = f.create_dataset(key, data=S_int8, compression='gzip', compression_opts=4)
                ds.attrs['class_idx'] = int(row['class_idx'])
                ds.attrs['species'] = row[LABEL_COL]
            except Exception as e:
                failed.append((idx, str(e)[:80]))
                if len(failed) < 5:
                    print(f'  Failed idx={idx}: {e}')

    print(f'\n✅ Cache built. Failed: {len(failed)} clips')
    !du -h $CACHE_PATH

Building cache → /content/drive/MyDrive/birdclef2026_anuj/melspec_cache.h5
Processing 35549 clips. This takes 30-60 min on free Colab...


Mel cache:   0%|          | 0/35549 [00:00<?, ?it/s]


✅ Cache built. Failed: 0 clips
967M	/content/drive/MyDrive/birdclef2026_anuj/melspec_cache.h5


In [ ]:
# Sanity-check the cache
with h5py.File(CFG['CACHE_H5'], 'r') as f:
    keys = list(f.keys())
    print(f'Total cached: {len(keys)}')

    # Sample one and visualize
    sample = f[keys[0]][:]
    print(f'Sample shape: {sample.shape}')
    print(f'Sample dtype: {sample.dtype}')
    print(f'Sample range: [{sample.min()}, {sample.max()}]')

    # Distribution of classes in cache
    cls_counts = {}
    for k in keys[:5000]:
        c = f[k].attrs['class_idx']
        cls_counts[int(c)] = cls_counts.get(int(c), 0) + 1
    print(f'\nSample of 5000 cached: covers {len(cls_counts)} classes')

# Filter train_df to keep only successfully cached entries
with h5py.File(CFG['CACHE_H5'], 'r') as f:
    cached_indices = set(int(k) for k in f.keys())
train_df = train_df[train_df.index.isin(cached_indices)].reset_index(drop=False).rename(columns={'index': 'cache_key'})
print(f'\nFinal train_df: {len(train_df)} rows')

Total cached: 35549
Sample shape: (128, 313)
Sample dtype: int8
Sample range: [40, 127]

Sample of 5000 cached: covers 28 classes

Final train_df: 35549 rows


## ✂️ Step 9 — Stratified 5-fold split

Group by recording (XC_id) where possible so the same recording doesn't appear in both train and val.

In [ ]:
# Stratified k-fold by species
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=CFG['N_FOLDS'], shuffle=True, random_state=CFG['SEED'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_df, train_df['class_idx'])):
    train_df.loc[val_idx, 'fold'] = fold

print('Fold distribution:')
print(train_df['fold'].value_counts().sort_index())
print(f'\nClasses per fold (should be ~equal):')
for f in range(CFG['N_FOLDS']):
    n_cls = train_df[train_df['fold'] == f]['class_idx'].nunique()
    print(f'  Fold {f}: {n_cls} unique classes')

# Save fold assignments
train_df[['cache_key', 'class_idx', 'fold', LABEL_COL]].to_csv(f'{WORK_DIR}/folds.csv', index=False)

Fold distribution:
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64

Classes per fold (should be ~equal):
  Fold 0: 198 unique classes
  Fold 1: 198 unique classes
  Fold 2: 199 unique classes
  Fold 3: 199 unique classes
  Fold 4: 198 unique classes


## 🎨 Step 10 — PyTorch Dataset with augmentation

In [ ]:
class BirdMelDataset(Dataset):
    """Dataset that reads cached mel-spectrograms from HDF5."""
    def __init__(self, df, cache_path, n_classes, training=True, img_size=224, cfg=CFG):
        self.df = df.reset_index(drop=True)
        self.cache_path = cache_path
        self.n_classes = n_classes
        self.training = training
        self.img_size = img_size
        self.cfg = cfg
        self._h5 = None  # opened lazily per worker

    def __len__(self):
        return len(self.df)

    def _ensure_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.cache_path, 'r')

    def __getitem__(self, idx):
        self._ensure_h5()
        row = self.df.iloc[idx]
        key = str(int(row['cache_key']))

        # Read mel-spec (stored as int8)
        S = self._h5[key][:].astype(np.float32) / 127.0  # back to [0, 1] range

        # Resize to img_size × img_size
        S = self._resize(S, self.img_size)

        # Time shift augmentation (training)
        if self.training and np.random.rand() < 0.5:
            shift = np.random.randint(-self.img_size // 8, self.img_size // 8)
            S = np.roll(S, shift, axis=1)

        # SpecAugment (training)
        if self.training:
            S = self._spec_augment(S)

        # Build label (one-hot)
        label = np.zeros(self.n_classes, dtype=np.float32)
        label[int(row['class_idx'])] = 1.0

        # Single channel
        return torch.from_numpy(S).unsqueeze(0).float(), torch.from_numpy(label).float()

    def _resize(self, S, target_size):
        h, w = S.shape
        # Pad or crop time axis
        if w < target_size:
            pad = target_size - w
            S = np.pad(S, ((0, 0), (0, pad)), mode='constant')
        elif w > target_size:
            start = np.random.randint(0, w - target_size + 1) if self.training else (w - target_size) // 2
            S = S[:, start:start + target_size]
        # Pad/crop frequency axis
        if h < target_size:
            pad = target_size - h
            S = np.pad(S, ((0, pad), (0, 0)), mode='constant')
        elif h > target_size:
            S = S[:target_size, :]
        return S.astype(np.float32)

    def _spec_augment(self, S, n_freq=2, n_time=2):
        h, w = S.shape
        for _ in range(n_freq):
            f = np.random.randint(0, self.cfg['SPECAUG_FREQ_MASK'])
            f0 = np.random.randint(0, max(1, h - f))
            S[f0:f0 + f, :] = 0
        for _ in range(n_time):
            t = np.random.randint(0, self.cfg['SPECAUG_TIME_MASK'])
            t0 = np.random.randint(0, max(1, w - t))
            S[:, t0:t0 + t] = 0
        return S

# Mixup helper
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    mixed_y = torch.maximum(y, y[idx])  # max-label mixup (BirdCLEF convention)
    return mixed_x, mixed_y, lam

print('✅ Dataset & augmentation defined')

✅ Dataset & augmentation defined


## 🧠 Step 11 — EfficientNet-B0 model

In [ ]:
def create_model(model_name=CFG['MODEL_NAME'], n_classes=N_CLASSES, pretrained=True):
    """Create timm model adapted for 1-channel mel-spectrogram input."""
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        in_chans=1,  # mono mel-spectrogram
        num_classes=n_classes,
    )
    return model

# Test it
model = create_model().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {CFG["MODEL_NAME"]}')
print(f'Parameters: {n_params/1e6:.2f}M')

# Verify forward pass
dummy = torch.randn(2, 1, CFG['IMG_SIZE'], CFG['IMG_SIZE']).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Forward pass output shape: {out.shape}')
del model, dummy; torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model: tf_efficientnet_b0.ns_jft_in1k
Parameters: 4.31M
Forward pass output shape: torch.Size([2, 234])


## 🚂 Step 12 — Training function with checkpoint resume

Saves checkpoints every epoch to Google Drive. If Colab disconnects, just rerun this cell — it resumes from the latest checkpoint.

In [ ]:
def train_one_fold(fold, train_df, n_classes=N_CLASSES, cfg=CFG):
    """Train one fold of EfficientNet-B0. Resume-friendly."""
    print(f'\n{"="*70}\n🚀 Training Fold {fold}/{cfg["N_FOLDS"]-1}\n{"="*70}')

    # Data
    tr_df = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_df = train_df[train_df['fold'] == fold].reset_index(drop=True)
    print(f'Train: {len(tr_df)} clips | Val: {len(val_df)} clips')

    train_ds = BirdMelDataset(tr_df, cfg['CACHE_H5'], n_classes, training=True, img_size=cfg['IMG_SIZE'])
    val_ds = BirdMelDataset(val_df, cfg['CACHE_H5'], n_classes, training=False, img_size=cfg['IMG_SIZE'])

    train_loader = DataLoader(train_ds, batch_size=cfg['BATCH_SIZE'], shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg['BATCH_SIZE'] * 2, shuffle=False, num_workers=2, pin_memory=True)

    # Model + optimizer
    model = create_model().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['LR'], weight_decay=cfg['WEIGHT_DECAY'])

    # Cosine schedule with linear warmup
    def lr_lambda(step):
        total = cfg['EPOCHS'] * len(train_loader)
        warmup = cfg['WARMUP_EPOCHS'] * len(train_loader)
        if step < warmup:
            return step / max(1, warmup)
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = GradScaler()

    # Resume checkpoint if exists
    ckpt_path = f'{cfg["CHECKPOINTS"]}/fold{fold}_latest.pt'
    best_path = f'{cfg["CHECKPOINTS"]}/fold{fold}_best.pt'
    start_epoch = 0
    best_auc = 0.0
    patience_left = cfg['PATIENCE']
    history = []

    if os.path.exists(ckpt_path):
        print(f'⤴️  Resuming from {ckpt_path}')
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler'])
        scaler.load_state_dict(ck['scaler'])
        start_epoch = ck['epoch'] + 1
        best_auc = ck['best_auc']
        patience_left = ck['patience_left']
        history = ck.get('history', [])
        print(f'   epoch={start_epoch}, best_auc={best_auc:.4f}, patience={patience_left}')

    # Loss
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(start_epoch, cfg['EPOCHS']):
        # === Training ===
        model.train()
        train_loss = 0
        n_batches = 0
        pbar = tqdm(train_loader, desc=f'Ep {epoch} train', leave=False)

        for x, y in pbar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # Mixup
            if np.random.rand() < cfg['MIXUP_PROB']:
                x, y, _ = mixup_data(x, y, alpha=cfg['MIXUP_ALPHA'])

            optimizer.zero_grad()
            with autocast():
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['GRAD_CLIP'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

        train_loss /= n_batches

        # === Validation ===
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for x, y in tqdm(val_loader, desc=f'Ep {epoch} val', leave=False):
                x = x.to(device, non_blocking=True)
                with autocast():
                    logits = model(x)
                all_logits.append(torch.sigmoid(logits).float().cpu().numpy())
                all_labels.append(y.numpy())

        all_logits = np.concatenate(all_logits)
        all_labels = np.concatenate(all_labels)

        # Macro ROC-AUC (only classes with ≥1 positive in val)
        valid = all_labels.sum(0) > 0
        val_auc = roc_auc_score(all_labels[:, valid], all_logits[:, valid], average='macro')

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_auc': val_auc, 'lr': scheduler.get_last_lr()[0]})
        print(f'Ep {epoch:2d} | train_loss={train_loss:.4f} | val_auc={val_auc:.4f} | patience_left={patience_left}')

        # Save latest checkpoint
        torch.save({
            'fold': fold, 'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'best_auc': best_auc, 'patience_left': patience_left,
            'history': history,
            'cfg': cfg,
        }, ckpt_path)

        # Save best
        if val_auc > best_auc:
            best_auc = val_auc
            patience_left = cfg['PATIENCE']
            torch.save({
                'fold': fold, 'epoch': epoch,
                'model': model.state_dict(),
                'val_auc': val_auc,
                'cfg': cfg,
            }, best_path)
            print(f'   ✅ New best AUC {best_auc:.4f} — saved {best_path}')
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f'   🛑 Early stop at epoch {epoch}')
                break

    # Save history
    pd.DataFrame(history).to_csv(f'{cfg["LOG_DIR"]}/fold{fold}_history.csv', index=False)

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache(); gc.collect()

    print(f'\n🏁 Fold {fold} done. Best val AUC: {best_auc:.4f}')
    return best_auc

print('✅ Training function defined')

✅ Training function defined


## ▶️ Step 13 — Train Fold 0 (validate pipeline first)

This takes ~2-3 hours on free Colab T4. If Colab disconnects, just rerun this cell — it'll resume from the latest checkpoint.

In [ ]:
# Train Fold 0 first to validate the pipeline
fold0_auc = train_one_fold(fold=0, train_df=train_df)
print(f'\n{"="*70}\nFold 0 final val AUC: {fold0_auc:.4f}\n{"="*70}')

# Expected range: 0.85-0.92 val AUC on training data
# If you see < 0.80, something is wrong — check the troubleshooting cells below
# If you see > 0.85, pipeline is working — proceed to remaining folds


🚀 Training Fold 0/4
Train: 28439 clips | Val: 7110 clips


Ep 0 train:   0%|          | 0/444 [00:00<?, ?it/s]

Ep 0 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep  0 | train_loss=0.1681 | val_auc=0.5398 | patience_left=8
   ✅ New best AUC 0.5398 — saved /content/drive/MyDrive/birdclef2026_anuj/checkpoints/fold0_best.pt


Ep 1 train:   0%|          | 0/444 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240> 
 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
      self._shutdown_workers() ^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    if w.is_alive():^
^ ^ ^  ^   ^^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'
^ ^ ^ ^ 
  File "/usr/lib/py

Ep 21 | train_loss=0.0184 | val_auc=0.9561 | patience_left=2


Ep 22 train:   0%|          | 0/444 [00:00<?, ?it/s]

Ep 22 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep 22 | train_loss=0.0179 | val_auc=0.9496 | patience_left=1
   🛑 Early stop at epoch 22

🏁 Fold 0 done. Best val AUC: 0.9562

Fold 0 final val AUC: 0.9562


## ⏯️ Step 14 — Train remaining folds (1-4)

Each fold takes ~2-3 hours. Run them across multiple Colab sessions. **Each fold cell is independently runnable** — if a session dies during fold 2, it resumes there next time.

In [ ]:
# Fold 1
fold1_auc = train_one_fold(fold=1, train_df=train_df)


🚀 Training Fold 1/4
Train: 28439 clips | Val: 7110 clips


Ep 0 train:   0%|          | 0/444 [00:00<?, ?it/s]

Ep 0 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep  0 | train_loss=0.1678 | val_auc=0.5095 | patience_left=8
   ✅ New best AUC 0.5095 — saved /content/drive/MyDrive/birdclef2026_anuj/checkpoints/fold1_best.pt


Ep 1 train:   0%|          | 0/444 [00:00<?, ?it/s]

...

Ep 25 train:   0%|          | 0/444 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>   
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
       self._shutdown_workers()   
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
     
if w.is_alive():^^ ^ ^^  ^ ^ ^ ^^^^^^^^^^^^^^^^^

Ep 25 val:   0%|          | 0/56 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
Exception ignored in:   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
if w.is_alive():Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
       self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
      ^if w.is_alive():^^
 ^ ^ ^  ^ ^ ^^^^^^^^
^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ ^ ^ ^ 
    File "/usr/lib/

Ep 25 | train_loss=0.0175 | val_auc=0.9491 | patience_left=1
   🛑 Early stop at epoch 25

🏁 Fold 1 done. Best val AUC: 0.9585


In [ ]:
# Fold 2
fold2_auc = train_one_fold(fold=2, train_df=train_df)


🚀 Training Fold 2/4
Train: 28439 clips | Val: 7110 clips


Ep 0 train:   0%|          | 0/444 [00:00<?, ?it/s]

Ep 0 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep  0 | train_loss=0.1689 | val_auc=0.5424 | patience_left=8
   ✅ New best AUC 0.5424 — saved /content/drive/MyDrive/birdclef2026_anuj/checkpoints/fold2_best.pt


Ep 1 train:   0%|          | 0/444 [00:00<?, ?it/s]

...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
        Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>  
^^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    ^if w.is_alive():^
 ^ ^ ^ ^ ^ ^ ^^^^^^^^^

Ep 25 | train_loss=0.0176 | val_auc=0.9472 | patience_left=2


Ep 26 train:   0%|          | 0/444 [00:00<?, ?it/s]

Ep 26 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep 26 | train_loss=0.0155 | val_auc=0.9454 | patience_left=1
   🛑 Early stop at epoch 26

🏁 Fold 2 done. Best val AUC: 0.9540


In [ ]:
# Fold 3
fold3_auc = train_one_fold(fold=3, train_df=train_df)


🚀 Training Fold 3/4
Train: 28439 clips | Val: 7110 clips


Ep 0 train:   0%|          | 0/444 [00:00<?, ?it/s]

Ep 0 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep  0 | train_loss=0.1679 | val_auc=0.5343 | patience_left=8
   ✅ New best AUC 0.5343 — saved /content/drive/MyDrive/birdclef2026_anuj/checkpoints/fold3_best.pt


Ep 1 train:   0%|          | 0/444 [00:00<?, ?it/s]

...

Ep 25 train:   0%|          | 0/444 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
self._shutdown_workers()
      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
self._shutdown_workers()    
if w.is_alive():  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    
if w.is_alive(): 
           ^  ^^^^^^^^^^^^^^^^^^^^
^^  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    
assert self._parent_pid == os.getpid(), 'can only test a child process'  File "/usr/lib/python3

Ep 25 val:   0%|          | 0/56 [00:00<?, ?it/s]

Exception ignored in: Traceback (most recent call last):
<function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
^^Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^    ^self._shutdown_workers()^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^^    if w.is_alive():
  ^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._parent_pid == os.getpid(), 'can only test a child process' 
^ ^^ ^^ ^ ^^  ^ ^^   ^
 ^  File "/us

Ep 25 | train_loss=0.0171 | val_auc=0.9448 | patience_left=1
   🛑 Early stop at epoch 25

🏁 Fold 3 done. Best val AUC: 0.9525


In [ ]:
# Fold 4
fold4_auc = train_one_fold(fold=4, train_df=train_df)


🚀 Training Fold 4/4
Train: 28440 clips | Val: 7109 clips


Ep 0 train:   0%|          | 0/444 [00:00<?, ?it/s]

Ep 0 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep  0 | train_loss=0.1713 | val_auc=0.5226 | patience_left=8
   ✅ New best AUC 0.5226 — saved /content/drive/MyDrive/birdclef2026_anuj/checkpoints/fold4_best.pt


Ep 1 train:   0%|          | 0/444 [00:00<?, ?it/s]

...

Ep 20 | train_loss=0.0178 | val_auc=0.9540 | patience_left=2


Ep 21 train:   0%|          | 0/444 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
^AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b96a88d7240>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Ep 21 val:   0%|          | 0/56 [00:00<?, ?it/s]

Ep 21 | train_loss=0.0173 | val_auc=0.9546 | patience_left=1
   🛑 Early stop at epoch 21

🏁 Fold 4 done. Best val AUC: 0.9610


## 📦 Step 15 — Export trained models to ONNX

ONNX runs ~3-5× faster on CPU than raw PyTorch. Required for the Kaggle 90-min CPU constraint.

In [ ]:
def export_fold_to_onnx(fold, cfg=CFG):
    """Export the best checkpoint of a fold to ONNX format."""
    best_path = f'{cfg["CHECKPOINTS"]}/fold{fold}_best.pt'
    onnx_path = f'{cfg["ONNX_DIR"]}/effnet_b0_fold{fold}.onnx'

    if not os.path.exists(best_path):
        print(f'⚠️  fold {fold} best checkpoint missing — skipping')
        return None

    print(f'Exporting fold {fold}...')
    ck = torch.load(best_path, map_location='cpu')

    model = create_model(pretrained=False).cpu()
    model.load_state_dict(ck['model'])
    model.eval()

    dummy = torch.randn(1, 1, cfg['IMG_SIZE'], cfg['IMG_SIZE'])

    torch.onnx.export(
        model, dummy, onnx_path,
        input_names=['input'], output_names=['logits'],
        dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=17, do_constant_folding=True,
    )

    # Verify the ONNX model
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    out = sess.run(None, {'input': dummy.numpy()})[0]
    print(f'  ✅ {onnx_path}: shape={out.shape}, file_size={os.path.getsize(onnx_path)/1024/1024:.1f}MB')
    return onnx_path

# Export all available folds
for fold in range(CFG['N_FOLDS']):
    export_fold_to_onnx(fold)

onnx_dir = CFG['ONNX_DIR']
print(f'\n✅ ONNX export complete. Files in {onnx_dir}:')
!ls -lh "$onnx_dir"

## ☁️ Step 16 — Upload trained ONNX models as Kaggle dataset

Required to use these models in the Kaggle inference notebook.

In [ ]:
# Create the dataset folder structure
import json
KAGGLE_DS_DIR = f'{WORK_DIR}/kaggle_dataset'
os.makedirs(KAGGLE_DS_DIR, exist_ok=True)

# Copy ONNX files into dataset folder
!cp "$WORK_DIR"/onnx/*.onnx "$KAGGLE_DS_DIR"/

# Save species list (needed by inference notebook to align outputs)
with open(f'{KAGGLE_DS_DIR}/species.json', 'w') as f:
    json.dump(SPECIES, f)

# Save config used for training (for reproducibility)
with open(f'{KAGGLE_DS_DIR}/training_config.json', 'w') as f:
    json.dump({k: v for k, v in CFG.items() if isinstance(v, (int, float, str, bool, list, dict))}, f, indent=2)

# Create dataset metadata for Kaggle API
kaggle_meta = {
    'title': 'BirdCLEF 2026 - Anuj EfficientNet-B0 ONNX',
    'id': 'anujdevsingh/birdclef-2026-anuj-effnet-b0-onnx',  # <-- REPLACE with your Kaggle username
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(f'{KAGGLE_DS_DIR}/dataset-metadata.json', 'w') as f:
    json.dump(kaggle_meta, f, indent=2)

print('Dataset folder contents:')
!ls -lh "$KAGGLE_DS_DIR"
print('\n⚠️  IMPORTANT: edit the "id" field in dataset-metadata.json with YOUR Kaggle username before next cell')